In [4]:
from Bio import Entrez
import arxiv
from rapidfuzz import fuzz
import pandas as pd
import time

Entrez.email = "zuhashaik12@gmail.com"  # Required by NCBI

def fetch_pubmed_metadata(pmid):
    handle = Entrez.efetch(db="pubmed", id=pmid, retmode="xml")
    records = Entrez.read(handle)
    try:
        article = records['PubmedArticle'][0]['MedlineCitation']['Article']
        title = article['ArticleTitle']
        authors = article.get('AuthorList', [])
        first_author = authors[0]['LastName'] if authors else "Unknown"
        return {"pmid": pmid, "title": title, "first_author": first_author}
    except Exception as e:
        print(f"Error fetching PMID {pmid}: {e}")
        return None

def search_arxiv_by_title(title):
    search = arxiv.Search(
        query=title,
        max_results=5,
        sort_by=arxiv.SortCriterion.Relevance
    )
    return list(search.results())

def match_paper(pmid_info, arxiv_results):
    best_score = 0
    best_arxiv_id = None
    for result in arxiv_results:
        score = fuzz.partial_ratio(pmid_info['title'].lower(), result.title.lower())
        if score > best_score and score > 85:  # adjustable threshold
            best_score = score
            best_arxiv_id = result.entry_id.split("/")[-1]
    return best_arxiv_id

def pmid_to_arxiv(pmid_list):
    mappings = []
    for pmid in pmid_list:
        meta = fetch_pubmed_metadata(pmid)
        if meta is None:
            mappings.append((pmid, None))
            continue
        arxiv_candidates = search_arxiv_by_title(meta["title"])
        arxiv_id = match_paper(meta, arxiv_candidates)
        mappings.append((pmid, arxiv_id))
        time.sleep(1)  # avoid rate limits
    return pd.DataFrame(mappings, columns=["pmid", "arxiv_id"])

In [ ]:
import pandas as pd
df = pd.read_csv('/Users/zuhashaik/Research/SUN-lab/datasets/drugcombo-data/dose_level.csv')
unique_pmids = df['PMID'].unique()[:100]
len(unique_pmids)

100

In [7]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
from Bio import Entrez
from urllib.parse import urljoin
import time
from tqdm import tqdm
# Set your Entrez email
Entrez.email = "zuhashaik12@gmail.com"

# Create the output directory
output_dir = "pdfs_dataset"
os.makedirs(output_dir, exist_ok=True)

# Function to get DOI from PubMed
def get_doi_from_pmid(pmid):
    try:
        handle = Entrez.efetch(db="pubmed", id=pmid, retmode="xml")
        records = Entrez.read(handle)
        handle.close()
        record = records['PubmedArticle'][0]
        
        # Try from ArticleIdList
        for aid in record['PubmedData']['ArticleIdList']:
            if aid.attributes['IdType'] == 'doi':
                return str(aid)
        
        # Try from ELocationID
        elocation = record['MedlineCitation']['Article']['ELocationID']
        for item in elocation:
            if item.attributes.get('EIdType') == 'doi':
                return str(item)
    except Exception as e:
        print(f"[{pmid}] DOI extraction failed: {e}")
    return None

# Function to download PDF
def download_pdf(pmid, doi):
    try:
        doi_url = f"https://doi.org/{doi}"
        response = requests.get(doi_url, headers={"User-Agent": "Mozilla/5.0"}, allow_redirects=True, timeout=15)
        soup = BeautifulSoup(response.text, "html.parser")

        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "pdf" in href.lower():
                pdf_link = urljoin(response.url, href)
                pdf_response = requests.get(pdf_link, timeout=15)
                if pdf_response.status_code == 200:
                    file_path = os.path.join(output_dir, f"{pmid}.pdf")
                    with open(file_path, 'wb') as f:
                        f.write(pdf_response.content)
                    print(f"[{pmid}] ✅ PDF downloaded.")
                    return
        print(f"[{pmid}] ❌ PDF link not found.")
    except Exception as e:
        print(f"[{pmid}] ❌ Download error: {e}")

# Loop over the DataFrame
for id in tqdm(unique_pmids):
    pmid = str(id)
    pdf_path = os.path.join(output_dir, f"{pmid}.pdf")
    if os.path.exists(pdf_path):
        print(f"[{pmid}] Already downloaded.")
        continue
    
    print(f"[{pmid}] Processing...")
    doi = get_doi_from_pmid(pmid)
    if doi:
        print(f"[{pmid}] Found DOI: {doi}")
        download_pdf(pmid, doi)
        time.sleep(1)  # polite delay
    else:
        print(f"[{pmid}] ❌ DOI not found.")


  0%|          | 0/100 [00:00<?, ?it/s]

[27853996] Processing...


/opt/anaconda3/envs/zuhas/lib/python3.12/site-packages/Bio/Entrez/Parser.py:1077: UserWarning: Failed to save pubmed_250101.dtd at /Users/zuhashaik/.config/biopython/Bio/Entrez/DTDs/pubmed_250101.dtd
  warnings.warn(f"Failed to save {filename} at {path}")


[27853996] Found DOI: 10.1007/s10637-016-0399-7
[27853996] ❌ PDF link not found.


  1%|          | 1/100 [00:11<19:15, 11.67s/it]

...

 98%|█████████▊| 98/100 [10:30<00:23, 11.82s/it]

[29764853] Processing...
[29764853] Found DOI: 10.1158/1078-0432.CCR-17-3716
[29764853] ❌ PDF link not found.


 99%|█████████▉| 99/100 [10:33<00:09,  9.27s/it]

[28546581] Processing...
[28546581] Found DOI: 10.1038/leu.2017.159
[28546581] ✅ PDF downloaded.


100%|██████████| 100/100 [10:45<00:00,  6.45s/it]
